<a href="https://colab.research.google.com/github/ext-colorful/LangChain-Essentials/blob/main/%F0%9F%A7%B1Building_Blocks%F0%9F%A7%B1L3_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[课程地址](https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388314-lesson-3-streaming)😁
[源码地址](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/L1_fast_agent.ipynb)😁
[LANGSMITH官网](https://smith.langchain.com)😁
[LANGCHAIN智能助手](https://chat.langchain.com)😁
[免费的智能体代理商](https://api.chatanywhere.tech)

In Lessons 2–7, you will learn how to use some of the fundamental building blocks in LangChain. These lessons explain and complement create_agent, and you’ll find them useful when creating your own agents. Each lesson is concise and focused.
> 在课程2-7中，你将学习如何使用LangChain中的一些基本构建模块。这些课程解释并补充了create_agent，当你创建自己的代理时，你会发现它们很有用。每个课程都简洁而专注。

## Learn how to reduce user-perceived latency using streaming.
> 学习如何使用流式传输来减少用户感知的延迟。

## Streaming

![链接文字](https://raw.githubusercontent.com/langchain-ai/lca-langchainV1-essentials/cc2313531a35124ca35f117e825365f2aadc0d5a/python/assets/LC_streaming.png)

Streaming reduces the latency between generating data and the user receiving it. There are two types frequently used with Agents:

## Setup
Load and/or check for needed environmental variables
> 加载和/或检查所需的环境变量

In [ ]:
!pip install -U langgraph>=1.0.0 langchain>=1.0.0 langchain-openai>=1.0.0 langchain-anthropic>=1.0.0 langchain-community>=0.4.0 langgraph-cli[inmem]>=0.4.0 langchain-mcp-adapters
# Used to securely store your API key
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('OPENAI_API_BASE')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain_agent_L3_streaming"

## Human👨‍💻 and AI 🤖 Messages

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="You are a full-stack comedian",
)

### No Steaming (invoke)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


Sure! Here’s one:

I told my computer I needed a break, and now it won’t stop sending me Kit-Kat ads. I guess even my hard drive’s got a sweet-tooth for boundary issues.


### values
You have seen this streaming mode in our examples so far.
> 到目前为止，你已经在我们的示例中见过这种流式模式。

In [ ]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Sure thing! Why do programmers prefer dark mode?

Because light attracts bugs.


### messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.
> 消息以令牌为单位逐个传输数据流，延迟极低。这非常适合聊天机器人等交互式应用。


In [ ]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

Here’s a sunny, family-friendly poem for you:

Morning light slips through the blinds,
A chorus of birds in two-door kinds.
The kettle hums a friendly tune,
While tiny toes tap to the spoon.

Breakfast bowls with smiling code,
Cereal castles in bright, bold mode.
Spoons become brave little boats,
Sailing across a sea of oat.

Dad tells jokes that softly land,
Mom bakes sunshine in a pan.
Siblings race with sneakers bright,
Laughter sparkles, pure delight.

We tidy up with gentle might,
Put toys to bed at soft goodnights.
We help, we share, we learn, we grow,
In every moment, love will show.

Hand in hand, we face the day,
We find new wonders along the way.
Family ties a cozy thread,
Woven with care, in every tread.

When evening comes with stars in sight,
We tuck away the day’s small light.
 dreams drift in like friendly ghosts,
And tomorrow brings another toast.

So here’s to hearts that feel just right,
To giggles tucked in soft moonlight.
A family, warm and brave and true,
Shared ad

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream custom data from sources you create.
> 流式传输通常是指在最终结果准备好之前向用户传递信息。有许多情况下这是很有用的。一个 get_stream_writer 写入器允许你轻松地从你创建的源流式传输自定义数据。



In [ ]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    # 流式传输任意数据
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[get_weather],
    system_prompt="You are a tool-using agent. For ANY weather-related user question you MUST call the tool named 'get_weather' and must NOT answer directly yourself."
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b4b14c28-ef20-4e05-9fd2-cdb228631f97')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='b4b14c28-ef20-4e05-9fd2-cdb228631f97'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 166, 'total_tokens': 190, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano', 'system_fingerprint': None, 'id': 'chatcmpl-CedI0aycmln26o8ngTPNCr0ZwdfVf', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--ee17bb61-16c2-48dc-981f-90fa2f0fd9dc-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, '

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')


## Try different modes on your own! 尝试自己使用不同的模式！
Modify the stream mode and the select to produce different results.
> 修改流模式和选择以产生不同的结果。

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

In [ ]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


## 自定义（custom）流没出现

### 改了get_weather
> 目前工具 docstring 太简短，请写清楚“必须使用工具”。例如：
```
def get_weather(city: str) -> str:
    """
    Get weather for a given city.

    This function MUST be used to get weather information.
    The agent should not answer weather questions directly.
    """
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

```
加了 system_prompt
```
agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[get_weather],
    system_prompt="You are a tool-using agent. For ANY weather-related user question you MUST call the tool named 'get_weather' and must NOT answer directly yourself."
)
```